# Leitura de Múltiplos Parquets com PySpark

Notebook para ler todos os arquivos Parquet de uma pasta, detectar se contêm dados JSON e expandir automaticamente.

## 1. Configurar SparkSession

Importar PySpark e criar uma SparkSession.

In [1]:
from pyspark.sql import SparkSession

# Criar SparkSession
spark = SparkSession.builder \
    .appName("Leitura Parquet") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
spark

Spark version: 3.4.0


## 2. Resumo dos Arquivos

Tabela resumo mostrando quais arquivos têm JSON e quantos registros.

In [2]:
import glob
import os

# Pasta com os arquivos Parquet
pasta_parquet = "quickbooks_data/daily_incremental/bill"

# Listar todos os arquivos .parquet
arquivos = glob.glob(os.path.join(pasta_parquet, "*.parquet"))
print(f"Arquivos encontrados: {len(arquivos)}\n")

# Dicionário para guardar os DataFrames
dataframes = {}

for arquivo in arquivos:
    nome = os.path.basename(arquivo)
    print(f"--- {nome} ---")
    df = spark.read.parquet(arquivo)
    registros = df.count()
    print(f"  Registros: {registros}")
    print(f"  Colunas: {df.columns}")

    # Detectar se alguma coluna contém JSON (string que começa com '{' ou '[')
    tem_json = False
    colunas_json = []
    for c in df.columns:
        # Verificar tipo string
        tipo = dict(df.dtypes).get(c)
        if tipo == "string" and registros > 0:
            amostra = df.select(c).filter(f"`{c}` IS NOT NULL").first()
            if amostra:
                valor = amostra[0]
                if isinstance(valor, str) and len(valor) > 0 and valor.strip()[0] in ('{', '['):
                    tem_json = True
                    colunas_json.append(c)

    if tem_json:
        print(f"  ✅ Contém JSON nas colunas: {colunas_json}")
    else:
        print(f"  ❌ Não contém dados JSON")

    dataframes[nome] = {"df": df, "tem_json": tem_json, "colunas_json": colunas_json}
    print()

Arquivos encontrados: 3

--- Bill_20260207170542.parquet ---
  Registros: 2
  Colunas: ['raw_value']
  ❌ Não contém dados JSON

--- Bill_20260325043100.parquet ---
  Registros: 10
  Colunas: ['raw_value']
  ✅ Contém JSON nas colunas: ['raw_value']

--- placeholder_bill_20260324.parquet ---
  Registros: 1
  Colunas: ['raw_value']
  ✅ Contém JSON nas colunas: ['raw_value']



import pandas as pd

resumo = []
for nome, info in dataframes.items():
    resumo.append({
        "Arquivo": nome,
        "Registros": info["df"].count(),
        "Colunas": len(info["df"].columns),
        "Tem JSON": "Sim" if info["tem_json"] else "Não",
        "Colunas JSON": ", ".join(info["colunas_json"]) if info["colunas_json"] else "-"
    })

df_resumo = pd.DataFrame(resumo)
df_resumo

## 3. Verificar se as Colunas JSON são Iguais

Comparar os schemas JSON de todos os arquivos que contêm JSON para verificar compatibilidade.

In [3]:
from pyspark.sql.functions import from_json, schema_of_json, col

# Coletar schemas JSON de cada arquivo
schemas_json = {}
for nome, info in dataframes.items():
    if not info["tem_json"]:
        continue
    df = info["df"]
    for col_json in info["colunas_json"]:
        amostra = df.select(col_json).filter(f"`{col_json}` IS NOT NULL").first()[0]
        json_schema = schema_of_json(amostra)
        # Parsear para obter o StructType real
        df_temp = df.withColumn("parsed", from_json(col(col_json), json_schema)).select("parsed.*")
        schemas_json[nome] = {
            "colunas": sorted(df_temp.columns),
            "schema": df_temp.schema,
            "dtypes": sorted(df_temp.dtypes, key=lambda x: x[0])
        }

# Comparar todos os schemas entre si
arquivos_com_json = list(schemas_json.keys())
print(f"Arquivos com JSON: {len(arquivos_com_json)}\n")

if len(arquivos_com_json) < 2:
    print("⚠️ Apenas 1 arquivo com JSON encontrado, nada para comparar.")
else:
    referencia = arquivos_com_json[0]
    colunas_ref = set(schemas_json[referencia]["colunas"])
    todos_iguais = True

    for arquivo in arquivos_com_json[1:]:
        colunas_atual = set(schemas_json[arquivo]["colunas"])

        print(f"Comparando: {referencia} vs {arquivo}")

        # Colunas iguais?
        if colunas_ref == colunas_atual:
            print(f"  ✅ Mesmas colunas ({len(colunas_ref)} colunas)")
        else:
            todos_iguais = False
            apenas_ref = colunas_ref - colunas_atual
            apenas_atual = colunas_atual - colunas_ref
            em_comum = colunas_ref & colunas_atual
            print(f"  ❌ Colunas diferentes!")
            print(f"     Em comum: {len(em_comum)} → {sorted(em_comum)}")
            if apenas_ref:
                print(f"     Só em {referencia}: {sorted(apenas_ref)}")
            if apenas_atual:
                print(f"     Só em {arquivo}: {sorted(apenas_atual)}")

        # Tipos iguais?
        dtypes_ref = schemas_json[referencia]["dtypes"]
        dtypes_atual = schemas_json[arquivo]["dtypes"]
        if dtypes_ref == dtypes_atual:
            print(f"  ✅ Tipos de dados idênticos")
        else:
            todos_iguais = False
            print(f"  ❌ Tipos de dados diferem:")
            for (col_r, tipo_r), (col_a, tipo_a) in zip(dtypes_ref, dtypes_atual):
                if col_r == col_a and tipo_r != tipo_a:
                    print(f"     Coluna '{col_r}': {tipo_r} → {tipo_a}")
        print()

    if todos_iguais:
        print("✅ RESULTADO: Todos os arquivos JSON têm exatamente as mesmas colunas e tipos!")
    else:
        print("❌ RESULTADO: Existem diferenças entre os schemas JSON dos arquivos.")

Arquivos com JSON: 2

Comparando: Bill_20260325043100.parquet vs placeholder_bill_20260324.parquet
  ✅ Mesmas colunas (19 colunas)
  ✅ Tipos de dados idênticos

✅ RESULTADO: Todos os arquivos JSON têm exatamente as mesmas colunas e tipos!


In [4]:
from pyspark.sql.functions import from_json, schema_of_json, col
from pyspark.sql.types import StructType, StringType

def flatten_df(df, sep="_"):
    """Achata recursivamente colunas StructType em colunas simples."""
    colunas_complexas = True
    while colunas_complexas:
        colunas_complexas = False
        novas_colunas = []
        for field in df.schema.fields:
            if isinstance(field.dataType, StructType):
                colunas_complexas = True
                for sub in field.dataType.fields:
                    novo_nome = f"{field.name}{sep}{sub.name}"
                    novas_colunas.append(df[field.name][sub.name].alias(novo_nome))
            else:
                novas_colunas.append(df[field.name])
        df = df.select(novas_colunas)
    return df

def parse_nested_json(df, sep="_", max_depth=3):
    """Detecta e parseia colunas string que contêm JSON, depois achata."""
    for _ in range(max_depth):
        str_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
        if not str_cols:
            break

        amostra = df.select(*[df[c] for c in str_cols]).filter(
            df[str_cols[0]].isNotNull()
        ).first()
        if not amostra:
            break

        cols_json = []
        for i, c in enumerate(str_cols):
            val = amostra[i]
            if val and isinstance(val, str) and val.strip() and val.strip()[0] in ('{', '['):
                cols_json.append((c, val))

        if not cols_json:
            break

        for c, val in cols_json:
            schema = schema_of_json(val)
            df = df.withColumn(c, from_json(df[c], schema))

        df = flatten_df(df, sep)
    return df

# Para cada arquivo com JSON, parsear, expandir sub-JSONs e achatar
for nome, info in dataframes.items():
    if not info["tem_json"]:
        print(f"--- {nome}: Sem JSON, mostrando dados diretos ---")
        info["df"].show(5, truncate=False)
        print()
        continue

    df = info["df"]
    for col_json in info["colunas_json"]:
        print(f"--- {nome} | Coluna: {col_json} ---")
        amostra = df.select(col_json).filter(df[col_json].isNotNull()).first()[0]
        json_schema = schema_of_json(amostra)

        df_parsed = df.withColumn("parsed", from_json(df[col_json], json_schema)) \
                      .select("parsed.*")

        colunas_antes = set(df_parsed.columns)

        # Parsear sub-JSONs e achatar tudo
        df_flat = parse_nested_json(df_parsed)

        colunas_expandidas = [c for c in df_flat.columns if c not in colunas_antes]
        if colunas_expandidas:
            print(f"Colunas JSON aninhadas expandidas: {colunas_expandidas}")

        print(f"Total de colunas: {len(df_flat.columns)}")
        df_flat.printSchema()
        df_flat.show(5, truncate=False)

        info["df_parsed"] = df_flat
    print()

--- Bill_20260207170542.parquet: Sem JSON, mostrando dados diretos ---
+-------------+
|raw_value    |
+-------------+
|QueryResponse|
|time         |
+-------------+


--- Bill_20260325043100.parquet | Coluna: raw_value ---
Colunas JSON aninhadas expandidas: ['APAccountRef_name', 'APAccountRef_value', 'CurrencyRef_name', 'CurrencyRef_value', 'DepartmentRef_name', 'DepartmentRef_value', 'MetaData_CreateTime', 'MetaData_LastUpdatedTime', 'MetaData_source', 'TxnTaxDetail_TotalTax', 'VendorRef_name', 'VendorRef_value']
Total de colunas: 25
root
 |-- APAccountRef_name: string (nullable = true)
 |-- APAccountRef_value: string (nullable = true)
 |-- Balance: string (nullable = true)
 |-- CurrencyRef_name: string (nullable = true)
 |-- CurrencyRef_value: string (nullable = true)
 |-- DepartmentRef_name: string (nullable = true)
 |-- DepartmentRef_value: string (nullable = true)
 |-- DocNumber: string (nullable = true)
 |-- DueDate: string (nullable = true)
 |-- ExchangeRate: string (nullable 

In [5]:
from IPython.display import display

# Mostrar as 5 primeiras linhas da tabela final de cada arquivo
for nome, info in dataframes.items():
    if "df_parsed" in info:
        print(f"\n--- {nome} (5 primeiras linhas) ---")
        display(info["df_parsed"].limit(5).toPandas())


--- Bill_20260325043100.parquet (5 primeiras linhas) ---


,APAccountRef_name,APAccountRef_value,Balance,CurrencyRef_name,CurrencyRef_value,DepartmentRef_name,DepartmentRef_value,DocNumber,DueDate,ExchangeRate,...,MetaData_source,PrivateNote,SyncToken,TotalAmt,TxnDate,TxnTaxDetail_TotalTax,VendorRef_name,VendorRef_value,domain,sparse
0,Accounts Payable,acct-004,59.50,United States Dollar,USD,Clinical,dep-2,B-3001,2026-04-01,1,...,bill,Supply bill batch 1,201,287.50,2026-03-01,20.13,Clinical Vendor 1,vend-001,QBO,false
1,Accounts Payable,acct-004,69.00,United States Dollar,USD,Admin,dep-3,B-3002,2026-04-02,1,...,bill,Supply bill batch 2,202,325.00,2026-03-02,22.75,Clinical Vendor 2,vend-002,QBO,false
2,Accounts Payable,acct-004,78.50,United States Dollar,USD,Front Office,dep-1,B-3003,2026-04-03,1,...,bill,Supply bill batch 3,203,362.50,2026-03-03,25.38,Clinical Vendor 3,vend-003,QBO,false
3,Accounts Payable,acct-004,88.00,United States Dollar,USD,Clinical,dep-2,B-3004,2026-04-04,1,...,bill,Supply bill batch 4,204,400.00,2026-03-04,28.00,Clinical Vendor 4,vend-004,QBO,false
4,Accounts Payable,acct-004,97.50,United States Dollar,USD,Admin,dep-3,B-3005,2026-04-05,1,...,bill,Supply bill batch 5,205,437.50,2026-03-05,30.63,Clinical Vendor 5,vend-005,QBO,false



--- placeholder_bill_20260324.parquet (5 primeiras linhas) ---


,APAccountRef,Balance,CurrencyRef,DepartmentRef,DocNumber,DueDate,ExchangeRate,Id,Line,LinkedTxn,MetaData,PrivateNote,SyncToken,TotalAmt,TxnDate,TxnTaxDetail,VendorRef,domain,sparse
0,None,0,None,None,PH-BILL-1,2026-02-14,1,placeholder-bill-1,None,None,None,Placeholder bill,1,0,2026-02-07,None,None,QBO,false


In [6]:
# Encerrar SparkSession
spark.stop()
print("SparkSession encerrada.")

SparkSession encerrada.
